# Task 1: Train & Serialize

In [ ]:
import numpy as np
import pandas as pd
import joblib
import requests
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

## 1. Load the Iris dataset and split into 80/20 train/test

In [ ]:
iris = load_iris()
X = iris.data
y = iris.target
target_names = iris.target_names

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

## 2. Train a RandomForestClassifier

In [ ]:
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

## 3. Evaluate on the test set

In [ ]:
y_pred = clf.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

## 4. Save the trained model

In [ ]:
joblib.dump(clf, "model.joblib")
joblib.dump(target_names, "target_names.joblib")

## 5. Load the model back and verify predictions

In [ ]:
loaded_model = joblib.load("model.joblib")
loaded_pred = loaded_model.predict(X_test)
print(f"Predictions match: {np.array_equal(y_pred, loaded_pred)}")

# Task 3: Test the API

## 1. Health check

In [ ]:
try:
    response = requests.get("http://localhost:5000/health")
    print(f"Status Code: {response.status_code}")
    print(f"Response: {response.json()}")
except Exception as e:
    print(f"Error: {e}")

## 2. Single prediction

In [ ]:
sample = [5.1, 3.5, 1.4, 0.2]
try:
    response = requests.post("http://localhost:5000/predict", json={"features": sample})
    print(f"Status Code: {response.status_code}")
    print(f"Response: {response.json()}")
except Exception as e:
    print(f"Error: {e}")

## 3. Error handling

In [ ]:
# Missing features
response = requests.post("http://localhost:5000/predict", json={})
print(f"Missing features Status Code: {response.status_code}")
print(f"Response: {response.json()}")

# Wrong count
response = requests.post("http://localhost:5000/predict", json={"features": [1, 2, 3]})
print(f"Wrong count Status Code: {response.status_code}")
print(f"Response: {response.json()}")

# Non-numeric
response = requests.post("http://localhost:5000/predict", json={"features": ["a", "b", "c", "d"]})
print(f"Non-numeric Status Code: {response.status_code}")
print(f"Response: {response.json()}")

## 4. Batch prediction

In [ ]:
samples = X_test[:5].tolist()
response = requests.post("http://localhost:5000/predict_batch", json={"samples": samples})
print(f"Status Code: {response.status_code}")
print(f"Response: {response.json()}")

local_preds = [target_names[p] for p in clf.predict(X_test[:5])]
print(f"Match local predictions: {response.json()['predictions'] == local_preds}")

# Task 4: Reflection

### Production Deployment Changes
- **WSGI Server**: Use Gunicorn or uWSGI instead of Flask's built-in development server.
- **Containerization**: Use Docker to package the app and its dependencies.
- **Environment Variables**: Use .env or system variables for configuration.
- **HTTPS**: Secure the API using SSL/TLS.
- **Authentication**: Add API keys or OAuth2 for security.

### Model Versioning
- Use a naming convention like `model_v1.joblib`, `model_v2.joblib`.
- Use MLflow or DVC for model tracking and versioning.
- Implement an A/B testing strategy or canary deployments.

### Monitoring
- **Latencies**: Monitor how long predictions take.
- **Drift**: Check if the distribution of incoming features changes over time (Data Drift).
- **Error Rates**: Track 4xx and 5xx responses.
- **Accuracy**: Periodically evaluate against ground truth if available.

### Scaling to 1,000 RPS
- **Load Balancer**: Use Nginx or cloud-native load balancers (AWS ELB).
- **Horizontal Scaling**: Run multiple instances of the API across different nodes.
- **Asynchronous Processing**: Use Celery/Redis for long-running tasks.
- **Caching**: Cache frequent predictions if applicable.